# Notebook 03 — Family B Parser Prototype

This notebook prototyped the parser for multi-attribute worksheets such as Data.

## Design rule

The Data worksheet must be recognized as a valid Family B sheet and validated by the parser, but it should not be included in the normalized dataset that feeds the downstream DSS pipeline.

In [10]:
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'samples').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Could not find the repository root from the current notebook location.')

repo_root = find_repo_root(Path.cwd().resolve())
workbook_path = repo_root / 'samples' / 'Données Marché Boursier_Projet_IA_copy.xlsx'
print('Workbook exists:', workbook_path.exists())

Workbook exists: True


In [11]:
raw = pd.read_excel(workbook_path, sheet_name='Data', header=None)

# Show a compact, readable preview of the sheet
preview = raw.iloc[:12, :8].copy()
preview.columns = [f'Col {i}' for i in range(preview.shape[1])]
preview

,Col 0,Col 1,Col 2,Col 3,Col 4,Col 5,Col 6,Col 7
0,Code AMC,1229,1211,1095,1094,1258,1181,1093
1,CODE ISIN,MA0000012296,MA0000012114,MA0000010951,MA0000010944,MA0000012585,MA0000011819,MA0000010936
2,LIBELLE,AFMA,AFRIC INDUSTRIES SA,AFRIQUIA GAZ,AGMA,AKDITAL,ALLIANCES,ALUMINIUM DU MAROC
3,NaN,"MA0000012296,XX,CAS","MA0000012114,XX,CAS","MA0000010951,XX,CAS","MA0000010944,XX,CAS","MA0000012585,XX,CAS","MA0000011819,XX,CAS","MA0000010936,XX,CAS"
4,NaN,AFMA P,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,ALTHIGHMID,ALTLOWMID,BASK,BBID,PERF-1YR,PERF-5YRSANN,52W-LOW
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-07-21 00:00:00,NaN,NaN,1279,1231,NaN,NaN,NaN
8,2026-07-20 00:00:00,NaN,NaN,1279,1230,NaN,NaN,NaN
9,2026-07-17 00:00:00,NaN,NaN,1279,1200,NaN,NaN,NaN


## Parser responsibilities

1. Detect the worksheet as a Family B structure.
2. Validate its block layout and metadata.
3. Report structural inconsistencies without failing the whole pipeline.
4. Exclude its variables from the unified normalized market dataset used downstream.